In [1]:
import asyncio
from agent_framework import WorkflowBuilder, WorkflowContext, executor, AgentResponseUpdate
from agent_framework.observability import configure_otel_providers
from typing_extensions import Never
from utils import new_base_openai_agent, new_base_gemini_agent
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
from telemetry import configure_ipc_telemetry
from my_telemetry_consumer import show_telemetry

# Note: Workflow events are a live application stream; 
# exported traces describe operations that have finished and are usually sent in batches.
os.environ["OTEL_BSP_SCHEDULE_DELAY"] = "200"
configure_ipc_telemetry(show_telemetry)

In [3]:
@executor(id="upper_case")
async def upper_case(text: str, ctx: WorkflowContext[str, str]):
    await ctx.send_message(text.upper())
    await ctx.yield_output('Upper Case is done')

@executor(id="reverse_text")
async def reverse_text(text: str, ctx: WorkflowContext[Never, str]) -> None:
    await asyncio.sleep(1)
    await ctx.yield_output(text[::-1])

workflow = (
    WorkflowBuilder(start_executor=upper_case)
    .add_edge(upper_case, reverse_text)
    .build()
)

1793965247.0916996: spans - {'name': 'workflow.build', 'context': {'trace_id': '0xeaf1e71a3974eb231c2b6dd1688ce562', 'span_id': '0xe60b57008c41a6f0', 'trace_state': '[]'}, 'kind': 'SpanKind.INTERNAL', 'parent_id': None, 'start_time': '2026-11-06T11:40:46.899607Z', 'end_time': '2026-11-06T11:40:46.902774Z', 'status': {'status_code': 'UNSET'}, 'attributes': {'workflow_builder.name': 'WorkflowBuilder-1a2fed15-5846-451b-952c-449403777657', 'workflow.id': '0c1c31cd-76e8-4f18-be62-c5e8340a9887', 'workflow.definition': '{"name": "WorkflowBuilder-1a2fed15-5846-451b-952c-449403777657", "id": "0c1c31cd-76e8-4f18-be62-c5e8340a9887", "start_executor_id": "upper_case", "max_iterations": 100, "edge_groups": [{"id": "InternalEdgeGroup/d1c3bc90-e73d-4198-9251-7b30523cb389", "type": "InternalEdgeGroup", "edges": [{"source_id": "internal:upper_case", "target_id": "upper_case"}]}, {"id": "InternalEdgeGroup/092bf6d3-68b7-415b-81f7-95c032391dfb", "type": "InternalEdgeGroup", "edges": [{"source_id": "intern

In [4]:
async for event in workflow.run("hello world", stream=True):
    print(event)

WorkflowEvent(type='started')
WorkflowEvent(type='status', state=IN_PROGRESS)
WorkflowEvent(type='superstep_started', iteration=1)
WorkflowEvent(type='executor_invoked', executor_id='upper_case', data='hello world')
WorkflowEvent(type='output', executor_id='upper_case', data='Upper Case is done')
WorkflowEvent(type='executor_completed', executor_id='upper_case', data=['HELLO WORLD', 'Upper Case is done'])
WorkflowEvent(type='superstep_completed', iteration=1)
WorkflowEvent(type='superstep_started', iteration=2)
WorkflowEvent(type='executor_invoked', executor_id='reverse_text', data='HELLO WORLD')
1793965249.3811185: spans - {'name': 'edge_group.process InternalEdgeGroup', 'context': {'trace_id': '0x8b0496330bfb1e186fb3d53b14300cc7', 'span_id': '0x8d3e49a83cf161f7', 'trace_state': '[]'}, 'kind': 'SpanKind.INTERNAL', 'parent_id': '0x903d0c09d618d62e', 'start_time': '2026-11-06T11:40:49.344253Z', 'end_time': '2026-11-06T11:40:49.344311Z', 'status': {'status_code': 'UNSET'}, 'attributes': 

1793965250.4115593: spans - {'name': 'executor.process reverse_text', 'context': {'trace_id': '0x8b0496330bfb1e186fb3d53b14300cc7', 'span_id': '0xba5dfb3036082a06', 'trace_state': '[]'}, 'kind': 'SpanKind.INTERNAL', 'parent_id': '0x903d0c09d618d62e', 'start_time': '2026-11-06T11:40:49.345176Z', 'end_time': '2026-11-06T11:40:50.359681Z', 'status': {'status_code': 'UNSET'}, 'attributes': {'executor.id': 'reverse_text', 'executor.type': 'FunctionExecutor', 'message.type': 'MessageType.STANDARD', 'message.payload_type': 'WorkflowMessage'}, 'events': [], 'links': [{'context': {'trace_id': '0x8b0496330bfb1e186fb3d53b14300cc7', 'span_id': '0x4b49d864bab29500', 'trace_state': '[]'}, 'attributes': {}}], 'resource': {'attributes': {'telemetry.sdk.language': 'python', 'telemetry.sdk.name': 'opentelemetry', 'telemetry.sdk.version': '1.45.2', 'service.instance.id': '7d09f202-0528-48fa-a7b0-bc761fc608bd', 'microsoft.agent-framework': 'True', 'service.name': 'agent_framework', 'service.version': '1.2

## GenAI Semantic Conventions

MAF uses [OpenTelemetry GenAI semantic conventions](https://github.com/open-telemetry/semantic-conventions-genai). This provides conventions for attribute names and values.

https://github.com/open-telemetry/semantic-conventions-genai/blob/main/docs/gen-ai/execute-tool.md

## Telemetry 101

Telemetry generally has three signals:

| Signal | Purpose | Example |
|---|---|---|
| **Traces** | Follow an operation across components | Workflow → executor → model request |
| **Logs** | Record individual messages or occurrences | “Request failed” |
| **Metrics** | Aggregate numerical measurements | Request count, token usage, duration histogram |

A **trace** consists of related **spans**. Each span represents an operation with a start time, end time, attributes, and possibly errors or timestamped events.

Workflow events arrive in realtime, while spans are recorded and only send when they have completed. Child spans can export before the parent finishes. OpenTelemetry does not need to wait for the entire trace—only for each individual span to end.

**A span represents one operation. A trace is the collection of related spans describing a larger operation end to end.**

### Example

Running a workflow produces several spans:

```text
Trace: one workflow execution
│
└── workflow.run                    ← root span
    ├── executor.process upper_case ← child span
    │   └── message.send            ← child of the executor span
    └── executor.process reverse_text
```

Every line naming an operation is a **span**. The entire collection is the **trace**.

The root span represents the overall workflow operation; **it is not itself the whole trace**.

### What a span contains

A span records information about its particular operation:

- **Name:** what operation ran.
- **Start and end timestamps:** when it ran and how long it took.
- **Attributes:** details such as executor ID or model name.
- **Status:** for example, whether an error occurred.
- **Events:** timestamped annotations during the operation.
- **IDs:** which trace it belongs to and which span is its parent.

### How they connect

Consider simplified IDs:

| Span | `trace_id` | `span_id` | `parent_span_id` |
|---|---|---|---|
| Workflow | `T1` | `A` | None |
| Upper-case executor | `T1` | `B` | `A` |
| Message send | `T1` | `C` | `B` |
| Reverse-text executor | `T1` | `D` | `A` |

- The shared **`trace_id`** groups them into one trace.
- Each **`span_id`** identifies an individual operation.
- The **parent span ID** establishes the hierarchy.

Another independent workflow execution would normally have a different trace ID.

### What your IPC exporter receives

Your span exporter receives **individual completed spans, grouped into export batches**—not necessarily a complete trace.

```text
First export:   spans B and C
Later export:   spans D and A
```

The Go process can group them by `trace_id` and reconstruct their relationships using the parent IDs. A batch can also contain spans from multiple traces.

This explains the terminology in your payload:

```json
{
  "signal": "traces",
  "records": ["individual span records go here"]
}
```

“Traces” names the telemetry category; **spans are the records being transported**.


### What starts a new trace? 

The framework decides, but typically a new span without an active parent span will create a new trace.

Start a span
│
├── Valid parent context exists
│   → inherit its trace ID
│   → create a new span ID
│   → become a child of that parent
│
└── No valid parent context exists
    → generate a new trace ID
    → create a new span ID
    → become the root span

We can create our own traces with the following:

```python
from opentelemetry import trace

tracer = trace.get_tracer("my-application")

with tracer.start_as_current_span("handle_user_request"):
    async for event in workflow.run("hello world", stream=True):
        print(event)
```

### Can we speed it up?

Yes, two options:

The easiest is to set this envrionment variable, which tells the BatchProcessor to check for and flush completed spans every 200ms.

`os.environ["OTEL_BSP_SCHEDULE_DELAY"] = "200"`

The other option is more convulted and involves creating your own providers and configuring its processor

```python
from agent_framework.observability import (
    create_resource,
    enable_instrumentation,
)
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

provider = TracerProvider(
    resource=create_resource(service_name="python-ai-engine")
)

provider.add_span_processor(
    BatchSpanProcessor(
        IPCSpanExporter(),
        schedule_delay_millis=200,
        max_export_batch_size=128,
        max_queue_size=2048,
    )
)

trace.set_tracer_provider(provider)
enable_instrumentation()
```

The snippet above covers traces only. To replace the existing configuration completely, also register:

- Your log exporter with a LoggerProvider and BatchLogRecordProcessor.
- Your metric exporter with a MeterProvider and PeriodicExportingMetricReader.
